# tieuluan3 · Hệ đa agent

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [ ]:
PROJECT_NAME = 'tieuluan3'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


### `src.common.contracts`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
"""Offline Phase 22 specification guards, not a training or TEST executor.

jsonschema/referencing are already pinned by the shared uv lock. No network fetch
is registered: unknown external schema URIs fail closed.
"""

from __future__ import annotations

import csv

import hashlib

import json

import math

from datetime import datetime

from functools import lru_cache

from itertools import pairwise

from pathlib import Path

from jsonschema import Draft202012Validator, FormatChecker

from referencing import Registry, Resource

ROOT = Path(__file__).resolve().parents[2]

SCHEMAS = ROOT / "src/common/schemas/v3"

PROTOCOL = ROOT / "experiments/configs/v3/protocol_v3.json"

REQUIREMENT_IDS = {f"REQ-{number:02}" for number in range(1, 55)}



#### ContractError

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
class ContractError(ValueError):
    """A structural or semantic specification violation."""



#### read_json

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8-sig"))



#### canonical_sha256

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def canonical_sha256(value) -> str:
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()



#### load_schemas

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
@lru_cache(maxsize=1)
def load_schemas() -> dict:
    schemas = {
        path.name.removesuffix(".schema.json"): read_json(path) for path in SCHEMAS.glob("*.schema.json")
    }
    for schema in schemas.values():
        Draft202012Validator.check_schema(schema)
    return schemas



#### _finite

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _finite(value):
    if isinstance(value, float) and not math.isfinite(value):
        raise ContractError("All numeric values must be finite; undefined metrics use null with a reason")
    if isinstance(value, dict):
        for child in value.values():
            _finite(child)
    elif isinstance(value, list):
        for child in value:
            _finite(child)



#### _backend

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _backend(document):
    allowed = {"scratch": {"numpy"}, "keras": {"tensorflow", "torch"}, "pytorch": {"torch"}}
    if document["backend"] not in allowed[document["implementation"]]:
        raise ContractError("implementation/backend mismatch")



#### _dataset_family

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset_family(document):
    protocol = read_json(PROTOCOL)
    dataset = next((row for row in protocol["datasets"] if row["dataset_id"] == document["dataset_id"]), None)
    if dataset is None or dataset["chapter"] != protocol["families"][document["family"]]["chapter"]:
        raise ContractError("Model family/dataset mismatch")



#### _time

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _time(value):
    return datetime.fromisoformat(value)



#### _model

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _model(document):
    _backend(document)
    _dataset_family(document)
    policy = read_json(PROTOCOL)
    layers = document["layers"]
    eligible = [
        layer
        for layer in layers
        if layer["kind"] in policy["layer_policy"]["counted_kinds"] and layer["parameter_count"] > 0
    ]
    expected = policy["families"][document["family"]]
    if len(eligible) < expected["minimum_learnable_layers"]:
        raise ContractError("Too few learnable layers; pooling/normalization/time steps do not count")
    if [layer["kind"] for layer in eligible] != expected["blocks"]:
        raise ContractError("Learnable layer composition differs from the locked reference architecture")
    if sum(layer["parameter_count"] for layer in layers) != document["parameter_count"]:
        raise ContractError("Total parameter count does not match layers")
    functions = {function["id"]: function for function in document["functions"]}
    if len(functions) != len(document["functions"]) or len({layer["id"] for layer in layers}) != len(layers):
        raise ContractError("Duplicate function/layer IDs")
    if any(name not in functions for name in document["composition"]):
        raise ContractError("Composition references a missing function")
    for layer in layers:
        if layer["function_id"] not in functions:
            raise ContractError("Layer references a missing function")
    if document["family"] in {"rnn", "lstm"} and not document["recurrent_unroll"]:
        raise ContractError("Recurrent model needs a time/layer unroll explanation")



#### _split

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _split(document):
    partitions = document["partitions"]
    if {part["name"] for part in partitions} != {"train", "validation", "test"}:
        raise ContractError("Each partition must occur exactly once")
    if document["method"] == "temporal":
        for part in partitions:
            if (
                not part["target_start"]
                or not part["target_end"]
                or _time(part["target_start"]) > _time(part["target_end"])
            ):
                raise ContractError("Temporal partition needs a valid target time range")
        by_name = {part["name"]: part for part in partitions}
        times = [
            (_time(by_name[name]["target_start"]), _time(by_name[name]["target_end"]))
            for name in ("train", "validation", "test")
        ]
        if any(left[1] >= right[0] for left, right in pairwise(times)):
            raise ContractError("Temporal target time ranges overlap")



#### _run

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _run(document):
    _backend(document)
    _dataset_family(document)
    if document["track"] == "matched_subset" and not document["subset_sha256"]:
        raise ContractError("Matched-subset run needs its selected train ID hash")
    if document["split"] == "test" and not document["test_seal_id"]:
        raise ContractError("TEST metrics require a test seal ID")
    if _time(document["started_at"]) > _time(document["finished_at"]):
        raise ContractError("Run finish precedes start")
    efficiency = document["efficiency"]
    if efficiency["inference_p95_ms"] < efficiency["inference_p50_ms"]:
        raise ContractError("Inference p95 cannot be below p50")
    if document["device"] == "cpu" and efficiency["peak_vram_bytes"] != 0:
        raise ContractError("CPU run cannot claim GPU allocator memory")
    for metric in document["metrics"]:
        if (metric["value"] is None) != bool(metric["undefined_reason"]):
            raise ContractError("Undefined metric must use null and an explicit reason")



#### _dataset

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset(document):
    if document["status"] == "ready":
        if document["source"]["license_status"] != "approved":
            raise ContractError("Ready dataset requires approved license/access")
        if document["logical_size"]["records"] is None:
            raise ContractError("Ready dataset requires measured logical size")
        if document["kind"] != "task_view" and not document["files"]:
            raise ContractError("Ready dataset needs file hashes")
    if document["kind"] == "task_view" and not document["parent_dataset_ids"]:
        raise ContractError("Task view must reference source datasets")



#### _seal

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _seal(document):
    expected = 1 if document["state"] in {"evaluating", "evaluated", "failed_sealed"} else 0
    if document["evaluation_count"] != expected:
        raise ContractError("Seal state/evaluation count mismatch")
    if len(set(document["approved_run_ids"])) != len(document["approved_run_ids"]):
        raise ContractError("Seal approved run IDs must be unique")



#### _artifact

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _artifact(document):
    loaders = {
        "npz": "numpy_no_pickle",
        "keras": "keras_safe_mode",
        "torch_state_dict": "torch_weights_only",
        "legacy_trusted_joblib": "legacy_allowlist",
    }
    expected = loaders.get(document["serialization"], "data_only")
    if document["load_policy"] != expected:
        raise ContractError("Artifact serialization/load policy mismatch")
    if document["kind"] == "checkpoint" and not document["cpu_loadable"]:
        raise ContractError("Deployable checkpoint must support CPU-safe load")



#### _api

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _api(document):
    datasets = {row["dataset_id"]: row for row in read_json(PROTOCOL)["datasets"]}
    dataset = datasets.get(document["dataset_id"])
    if not dataset or dataset["chapter"] != document["chapter"]:
        raise ContractError("Request dataset/chapter mismatch")
    family = document["family"]
    chapter_families = {2: {"mlp"}, 3: {"cnn"}, 4: {"rnn", "lstm"}, 5: {"kg_rag"}, 6: {"multi_agent"}}
    if family not in chapter_families[document["chapter"]]:
        raise ContractError("Request family/chapter mismatch")
    expected = (
        ["B0", "B1", "B2", "B3", "B4"]
        if family == "kg_rag"
        else ["A0", "A1", "A2"]
        if family == "multi_agent"
        else ["scratch", "keras", "pytorch"]
    )
    if document["implementations"] != expected:
        raise ContractError("Request implementations do not match family")
    kind = document["input"]["kind"]
    if kind != {2: "tabular", 3: "image", 4: "sequence", 5: "question", 6: "question"}[document["chapter"]]:
        raise ContractError("Request payload kind does not match chapter")
    if kind == "sequence" and len({len(row) for row in document["input"]["values"]}) != 1:
        raise ContractError("Sequence rows must have a consistent feature width")



#### _job

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _job(document):
    results = document["results"]
    expected = (
        {"B0", "B1", "B2", "B3", "B4"}
        if document["family"] == "kg_rag"
        else {"A0", "A1", "A2"}
        if document["family"] == "multi_agent"
        else {"scratch", "keras", "pytorch"}
    )
    names = [result["implementation"] for result in results]
    if len(names) != len(set(names)) or any(name not in expected for name in names):
        raise ContractError("Duplicate or inapplicable result slot")
    if document["status"] in {"succeeded", "partial_failed", "failed"} and set(names) != expected:
        raise ContractError("Terminal job must report every implementation, including unavailable slots")
    all_success = len(results) == len(expected) and all(result["status"] == "succeeded" for result in results)
    if any(result["input_sha256"] != document["input_sha256"] for result in results):
        raise ContractError("Job result input hash differs from the shared input hash")
    groups = {result["comparison_group_sha256"] for result in results}
    if document["comparison_valid"] and (len(groups) != 1 or None in groups):
        raise ContractError("Models do not belong to the same comparison group")
    if document["comparison_valid"] and (not all_success or document["comparison_blockers"]):
        raise ContractError("Incomplete or blocked comparison cannot be valid")
    for result in results:
        if result["status"] == "succeeded" and (
            result["prediction"] is None
            or result["error"] is not None
            or result["actual_device"] == "not_run"
            or not result["artifact_sha256"]
        ):
            raise ContractError("Successful result needs prediction, actual device and artifact hash")
        if result["status"] != "succeeded" and (result["error"] is None or result["prediction"] is not None):
            raise ContractError("Failed result needs explicit error, not a fabricated prediction")
        prediction = result["prediction"]
        if (
            prediction
            and prediction["kind"] == "classification"
            and not math.isclose(sum(row["value"] for row in prediction["probabilities"]), 1.0, abs_tol=1e-5)
        ):
            raise ContractError("Classification probabilities must sum to one")



#### _agent

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent(document):
    seen = set()
    agents = set()
    for node in document["nodes"]:
        if node["node_id"] in seen or not set(node["parent_ids"]).issubset(seen):
            raise ContractError("Agent trace must be acyclic and topologically ordered")
        seen.add(node["node_id"])
        agents.add(node["agent"])
    required = {
        "A0": {"direct"},
        "A1": {"router", "specialist"},
        "A2": {"router", "specialist", "verifier", "reporter"},
    }
    if not agents.issubset(required[document["variant"]]):
        raise ContractError("Agent trace uses a role outside its benchmark variant")
    if document["status"] == "succeeded" and not required[document["variant"]].issubset(agents):
        raise ContractError("Agent trace omits required roles")



#### _agent_task

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent_task(document):
    dataset = next(
        (row for row in read_json(PROTOCOL)["datasets"] if row["dataset_id"] == document["dataset_id"]), None
    )
    if not dataset or dataset["chapter"] != 6:
        raise ContractError("Agent task must use a chapter6 task-view dataset")



#### validate_document

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def validate_document(name: str, document: dict, *, publish: bool = False) -> None:
    schema_name, _, definition = name.partition("#")
    schemas = load_schemas()
    if schema_name not in schemas or (definition and definition not in schemas[schema_name].get("$defs", {})):
        raise ContractError(f"Unknown schema: {name}")
    _finite(document)
    registry = Registry().with_resources(
        (schema["$id"], Resource.from_contents(schema)) for schema in schemas.values()
    )
    schema = (
        {"$ref": schemas[schema_name]["$id"] + f"#/$defs/{definition}"}
        if definition
        else schemas[schema_name]
    )
    validator = Draft202012Validator(schema, registry=registry, format_checker=FormatChecker())
    errors = sorted(validator.iter_errors(document), key=lambda error: error.json_path)
    if errors:
        raise ContractError("; ".join(f"{error.json_path}: {error.message}" for error in errors[:5]))
    if publish and document.get("evidence_kind") == "synthetic_fixture":
        raise ContractError("Cannot publish synthetic fixture as measured evidence")
    checks = {
        "dataset": _dataset,
        "split": _split,
        "model": _model,
        "run": _run,
        "seal": _seal,
        "artifact": _artifact,
        "api": _api,
        "api#job": _job,
        "api#agent_task": _agent_task,
        "agent": _agent,
    }
    if name in checks:
        checks[name](document)



#### check_seal_transition

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def check_seal_transition(seal: dict, target: str, *, config_sha256: str) -> None:
    """Validate a proposed transition; do not change files or authorize TEST execution."""
    validate_document("seal", seal)
    if seal["state"] in {"evaluated", "failed_sealed"}:
        raise ContractError("TEST is sealed; do not reopen it")
    if config_sha256 != seal["locked_config_sha256"]:
        raise ContractError("TEST config hash changed")
    allowed = {"draft": {"locked"}, "locked": {"evaluating"}, "evaluating": {"evaluated", "failed_sealed"}}
    if target not in allowed.get(seal["state"], set()):
        raise ContractError("Invalid TEST seal transition")



#### _refs

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _refs(value):
    if isinstance(value, dict):
        if "$ref" in value:
            yield value["$ref"]
        for child in value.values():
            yield from _refs(child)
    elif isinstance(value, list):
        for child in value:
            yield from _refs(child)



#### audit_specification

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def audit_specification(root: Path) -> dict:
    """Audit frozen design documents; never read dataset contents or execute training."""
    errors = []
    result = {"errors": errors, "requirements": 0, "datasets": 0}
    required = [
        "docs/srs_v3.md",
        "docs/traceability_v3.csv",
        "docs/report_style_v3.md",
        "docs/architecture_v3.mmd",
        "docs/risk_register_v3.csv",
        "docs/adr/0001-v3-sources-and-runtime.md",
        "docs/adr/0002-v3-comparison-and-seals.md",
        "docs/adr/0003-v3-api-and-serving.md",
        "docs/adr/0004-v3-formula-minimal-report.md",
        "experiments/configs/v3/protocol_v3.json",
        "report/outline_v3.json",
        "docs/contracts/v3/openapi.json",
    ]
    for relative in required:
        if not (root / relative).is_file():
            errors.append(f"Missing required specification: {relative}")
    if errors:
        return result
    with (root / "docs/traceability_v3.csv").open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    result["requirements"] = len(rows)
    if {row["requirement_id"] for row in rows} != REQUIREMENT_IDS or len(rows) != len(REQUIREMENT_IDS):
        errors.append("Requirement IDs are missing, duplicated or unknown")
    for row in rows:
        if not all(
            row.get(key)
            for key in (
                "origin",
                "requirement",
                "phases",
                "spec_artifact",
                "planned_artifact",
                "acceptance_test",
                "implementation_status",
            )
        ):
            errors.append(f"Incomplete traceability: {row.get('requirement_id')}")
        if not (root / row["spec_artifact"]).is_file():
            errors.append(f"Missing specification artifact for {row['requirement_id']}")
        if any(not token.isdigit() or not 20 <= int(token) <= 35 for token in row["phases"].split(";")):
            errors.append(f"Invalid phase mapping for {row['requirement_id']}")
    protocol = read_json(root / "experiments/configs/v3/protocol_v3.json")
    outline = read_json(root / "report/outline_v3.json")
    datasets = protocol["datasets"]
    result["datasets"] = len(datasets)
    if len(datasets) != 18 or len({row["dataset_id"] for row in datasets}) != 18:
        errors.append("Expected 18 unique dataset/benchmark/view IDs")
    if [row["chapter"] for row in outline["chapters"]] != list(range(1, 7)):
        errors.append("Outline must contain exactly six chapters")
    for chapter in outline["chapters"]:
        expected = {row["dataset_id"] for row in datasets if row["chapter"] == chapter["chapter"]}
        if len(expected) != 3 or set(chapter["dataset_ids"]) != expected:
            errors.append(f"Dataset/outline mismatch: chapter {chapter['chapter']}")
        deploy = [section for section in chapter["sections"] if section["kind"] == "deployment"]
        if chapter["chapter"] >= 2 and (len(deploy) != 1 or deploy[0] != chapter["sections"][-1]):
            errors.append("Deployment must be the last short section, never its own chapter")
    if protocol["ch5"]["new_test_evaluation_allowed"] or protocol["ch5"]["test_run_count"] != 1:
        errors.append("Ch5 TEST must remain sealed at run_count=1")
    if protocol["comparison"]["training_runs_required"] != 4 * 3 * 3 * len(protocol["seeds"]["stability"]):
        errors.append("Neural run matrix count mismatch")
    api_path = root / "docs/contracts/v3/openapi.json"
    api = read_json(api_path)
    if api.get("openapi") != "3.1.1":
        errors.append("OpenAPI version must be explicit")
    for reference in _refs(api):
        relative, _, fragment = reference.partition("#")
        target = (api_path.parent / relative).resolve() if relative else api_path.resolve()
        try:
            target.relative_to(root.resolve())
            content = read_json(target)
            for token in filter(None, fragment.split("/")):
                content = content[token.replace("~1", "/").replace("~0", "~")]
        except (ValueError, KeyError, OSError) as error:
            errors.append(f"Unresolved local API reference {reference}: {error}")
    return result



### `src.common.artifact_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
"""Data-only, hash-verified checkpoints and complete provenance (no pickle)."""

from __future__ import annotations

import hashlib

import io

import json

import re

import zipfile

from pathlib import Path

import numpy as np

from src.common.contracts import canonical_sha256



#### sha256_file

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



#### write_once

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def write_once(path, document):
    path = Path(path)
    encoded = (
        json.dumps(document, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"
    ).encode()
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("xb") as stream:
            stream.write(encoded)
    except FileExistsError:
        if path.read_bytes() != encoded:
            raise ValueError(f"Immutable artifact differs: {path.name}") from None
    return {"path": path, "sha256": hashlib.sha256(encoded).hexdigest()}



#### ArtifactStore

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
class ArtifactStore:
    PROVENANCE_KEYS = (
        "dataset_sha256",
        "split_sha256",
        "subset_sha256",
        "preprocessing_sha256",
        "adapter_sha256",
        "config_sha256",
        "code_sha256",
        "environment_sha256",
        "model_contract_sha256",
    )

    def __init__(self, root, max_bytes=128 * 1024**2):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = max_bytes

    def _path(self, name, suffix):
        if not re.fullmatch(r"[a-z][a-z0-9_-]{0,95}", name):
            raise ValueError("Invalid checkpoint ID")
        path = (self.root / (name + suffix)).resolve()
        if path.parent != self.root:
            raise ValueError("Checkpoint path escapes root")
        return path

    def _provenance(self, provenance):
        if set(provenance) != set(self.PROVENANCE_KEYS) or any(
            not isinstance(v, str) or not re.fullmatch(r"[a-f0-9]{64}", v) for v in provenance.values()
        ):
            raise ValueError("Missing/invalid artifact provenance")

    def save_checkpoint(self, name, arrays, provenance):
        self._provenance(provenance)
        path = self._path(name, ".npz")
        metadata_path = self._path(name, ".json")
        if path.exists() or metadata_path.exists():
            raise ValueError("Checkpoint already exists")
        if not arrays or any(
            not re.fullmatch(r"[a-z][a-z0-9_/-]{0,127}", k)
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Only finite named numeric arrays may be saved")
        if sum(v.nbytes for v in arrays.values()) > self.max_bytes:
            raise ValueError("Checkpoint size budget exceeded")
        buffer = io.BytesIO()
        np.savez(buffer, **arrays)
        payload = buffer.getvalue()
        if len(payload) > self.max_bytes:
            raise ValueError("Serialized size budget exceeded")
        metadata = {
            "checkpoint_version": "phase25.1",
            "serialization": "npz",
            "load_policy": "numpy_no_pickle",
            "cpu_loadable": True,
            "sha256": hashlib.sha256(payload).hexdigest(),
            "size_bytes": len(payload),
            "provenance": dict(provenance),
            "arrays": {k: {"shape": list(v.shape), "dtype": str(v.dtype)} for k, v in arrays.items()},
        }
        with path.open("xb") as stream:
            stream.write(payload)
        write_once(metadata_path, metadata)
        return metadata

    def load_checkpoint(self, name, expected_provenance, *, expected_manifest_sha256=None):
        self._provenance(expected_provenance)
        if (
            expected_manifest_sha256 is not None
            and sha256_file(self._path(name, ".json")) != expected_manifest_sha256
        ):
            raise ValueError("Checkpoint manifest hash mismatch")
        metadata = json.loads(self._path(name, ".json").read_text(encoding="utf-8"))
        path = self._path(name, ".npz")
        if (
            metadata["provenance"] != expected_provenance
            or metadata.get("checkpoint_version") != "phase25.1"
            or metadata.get("load_policy") != "numpy_no_pickle"
        ):
            raise ValueError("Checkpoint provenance/load policy mismatch")
        if (
            path.stat().st_size != metadata["size_bytes"]
            or path.stat().st_size > self.max_bytes
            or sha256_file(path) != metadata["sha256"]
        ):
            raise ValueError("Checkpoint integrity mismatch")
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != metadata["sha256"]:
            raise ValueError("Checkpoint changed while loading")
        with zipfile.ZipFile(io.BytesIO(payload)) as packed:
            if sum(item.file_size for item in packed.infolist()) > self.max_bytes:
                raise ValueError("Expanded checkpoint exceeds size budget")
        with np.load(io.BytesIO(payload), allow_pickle=False) as archive:
            arrays = {k: archive[k] for k in archive.files}
        if set(arrays) != set(metadata["arrays"]) or any(
            list(v.shape) != metadata["arrays"][k]["shape"]
            or str(v.dtype) != metadata["arrays"][k]["dtype"]
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Checkpoint tensor metadata mismatch")
        return arrays



#### code_fingerprint

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def code_fingerprint(root, paths):
    root = Path(root)
    return canonical_sha256({str(path): sha256_file(root / path) for path in sorted(paths)})



### `src.ch05_knowledge_rag.preservation`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
"""Read-only inventory and exact content guards for the migrated Chapter5 baseline."""

import csv

import gzip

import re

from pathlib import PurePosixPath

from src.common.artifact_core import sha256_file, write_once

from src.common.contracts import ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase30"

RUNTIME = "apps/api/models/kg_manifest.json"

RUNTIME_SHA = "8b58e70d6d892cd697e8174ccdbaad78bcd0098a9cd2bc23ca2fda5724823073"

SEAL = "artifacts/manifests/ch5/phase15_test_run.json"

SEAL_SHA = "64f4167eb039903d9a70f56cf6c24a761ec59ad6986faa03f866b97856c7b929"

LOCKED = "experiments/configs/ch5/phase15_locked.json"

LOCKED_SHA = "560c671a3920bd6bde9f7fc005f3c952f857c55969c989bb6109932623210c9e"



#### checked

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def checked(record, root=ROOT):
    name, digest = record["path"], record["sha256"]
    if (
        not isinstance(name, str)
        or not name
        or "\\" in name
        or ":" in name
        or PurePosixPath(name).is_absolute()
        or ".." in PurePosixPath(name).parts
        or not isinstance(digest, str)
        or not re.fullmatch("[a-f0-9]{64}", digest)
    ):
        raise ValueError("Unsafe artifact reference")
    path = (root / name).resolve()
    if not path.is_relative_to(root.resolve()) or not path.is_file():
        raise ValueError("Artifact missing or outside workspace")
    if sha256_file(path) != digest:
        raise ValueError("Artifact hash mismatch: " + name)
    return path



#### ref

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def ref(path, root=ROOT):
    return {"path": path.relative_to(root).as_posix(), "sha256": sha256_file(path)}



#### verify_records

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def verify_records(records, root=ROOT):
    seen = set()
    for row in records:
        if row["path"] in seen:
            raise ValueError("Duplicate artifact reference")
        checked(row, root)
        seen.add(row["path"])
    return len(seen)



#### historical_baseline

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def historical_baseline():
    counts = {"raw": 0, "config": 0, "metric": 0}
    mappings = (
        ("raw", "term_paper/knowledge_rag/data/raw/", "data/raw/ch5/"),
        ("config", "term_paper/knowledge_rag/config/", "experiments/configs/ch5/"),
        ("metric", "term_paper/knowledge_rag/artifacts/metrics/", "artifacts/metrics/ch5/"),
    )
    with gzip.open(
        ROOT / "artifacts/manifests/phase20/before-files.csv.gz", "rt", encoding="utf-8-sig", newline=""
    ) as stream:
        for row in csv.DictReader(stream):
            for kind, old, new in mappings:
                if row["path"].startswith(old):
                    checked({"path": new + row["path"][len(old) :], "sha256": row["sha256"]})
                    counts[kind] += 1
    if counts != {"raw": 11345, "config": 7, "metric": 23}:
        raise ValueError("Historical baseline inventory incomplete")
    checked({"path": SEAL, "sha256": SEAL_SHA})
    checked({"path": LOCKED, "sha256": LOCKED_SHA})
    seal = read_json(ROOT / SEAL)
    if seal["status"] != "COMPLETE" or seal["run_count"] != 1 or seal["locked_config_sha256"] != LOCKED_SHA:
        raise ValueError("Historical TEST seal changed")
    for record in seal["prediction_artifacts"]:
        checked(
            {
                "path": record["path"].replace(
                    "term_paper/knowledge_rag/artifacts/predictions/", "artifacts/predictions/ch5/"
                ),
                "sha256": record["sha256"],
            }
        )
    return {**counts, "sealed_predictions": len(seal["prediction_artifacts"]), "test_run_count": 1}



#### create_snapshot

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def create_snapshot():
    path = BASE / "preserved_snapshot.json"
    if path.exists():
        document = read_json(path)
        verify_records(document["files"])
        return document
    print("Phase30: checking original raw/config/metric/TEST baseline", flush=True)
    baseline = historical_baseline()
    print("Phase30: baseline PASS; inventory of preserved migrated assets", flush=True)
    folders = (
        "data/raw/ch5",
        "experiments/configs/ch5",
        "artifacts/manifests/ch5",
        "artifacts/metrics/ch5",
        "artifacts/predictions/ch5",
        "artifacts/figures/ch5",
        "src/term_paper/tools",
        "report/assets/ch5_knowledge_rag",
        "report/references/dataset_cards/ch5",
    )
    files = {
        p
        for folder in folders
        for p in (ROOT / folder).rglob("*")
        if p.is_file() and (folder == "data/raw/ch5" or "__pycache__" not in p.parts)
    }
    files.update((ROOT / "apps/api/models").glob("kg*"))
    files.update((ROOT / "report/references/dataset_cards").glob("ch5*.md"))
    for name in (
        "term_paper/__init__.py",
        "apps/api/knowledge_rag_service.py",
        "report/sections/05_knowledge_rag.md",
        "report/references/bibliography.bib",
        "uv.lock",
        "pyproject.toml",
        "artifacts/manifests/environment.json",
        "artifacts/manifests/phase29/experiment_lock.json",
        "releases/v2/Tieu_luan_AI_ML_CNN_RNN_KG_RAG.docx",
    ):
        files.add(ROOT / name)
    dependencies = []
    for name in ("phase22/spec_lock.json", "phase24/data_lock.json", "phase25/core_lock.json"):
        lock_path = ROOT / "artifacts/manifests" / name
        lock = read_json(lock_path)
        verify_records(lock["files"])
        dependencies.append({**ref(lock_path), "verified_files": len(lock["files"])})
        files.add(lock_path)
    document = {
        "phase": 30,
        "status": "PASS",
        "baseline": baseline,
        "dependencies": dependencies,
        "files": [ref(p) for p in sorted(files)],
        "new_test_evaluation_executed": False,
        "policy": "Read-only original bytes; no rebuild, tuning, public TEST rerun or reserialization",
    }
    write_once(path, document)
    return document



### `src.ch06_multi_agent.core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
"""Deterministic roles and bounded state transitions; gold labels never enter this module."""

import math

import re

import time

import unicodedata

from src.common.contracts import canonical_sha256, validate_document

DOMAINS = ("diabetes", "house", "gold")

VARIANTS = ("A0", "A1", "A2")

WARNING = "Học thuật trên snapshot cũ; không chẩn đoán, kê đơn, định giá hiện tại hoặc tư vấn đầu tư."

TOOLS = {
    ("diabetes", "numeric"): "predict_diabetes",
    ("house", "numeric"): "predict_house",
    ("gold", "numeric"): "forecast_gold",
    ("diabetes", "evidence"): "retrieve_medical",
    ("house", "evidence"): "lookup_series",
    ("gold", "evidence"): "lookup_series",
}



#### normalized

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
def normalized(text):
    text = unicodedata.normalize("NFKD", text.casefold()).replace("đ", "d")
    return " ".join("".join(c for c in text if not unicodedata.combining(c)).split())



#### route

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
def route(question):
    text = normalized(question)
    matches = [
        domain
        for domain, terms in {
            "diabetes": ("tieu duong", "diabetes", "cdc", "medquad"),
            "house": ("gia nha", "housing", "ames", "bis"),
            "gold": ("gia vang", "gold", "pink sheet"),
        }.items()
        if any(re.search(r"\b" + re.escape(term) + r"\b", text) for term in terms)
    ]
    return matches[0] if len(matches) == 1 else None



#### policy_reason

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
def policy_reason(question, operation):
    text = normalized(question)
    if any(
        term in text
        for term in (
            "ke don",
            "lieu insulin",
            "chan doan cho toi",
            "diagnose me",
            "prescribe",
            "chac chan sinh loi",
            "dam bao loi nhuan",
            "tu van dau tu",
            "guaranteed profit",
            "ignore previous",
            "reveal secrets",
            "bo qua quy tac",
            "tiet lo bi mat",
        )
    ):
        return "unsafe_request"
    return "unsupported_operation" if operation == "unsupported" else None



#### validate_task

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
def validate_task(task):
    keys = {"task_id", "domain_hint", "question", "record_id", "operation"}
    if not isinstance(task, dict) or set(task) != keys:
        raise ValueError("Require a bounded structured task, not arbitrary instructions")
    if task["domain_hint"] not in DOMAINS or task["operation"] not in {"numeric", "evidence", "unsupported"}:
        raise ValueError("Undeclared domain or operation")
    if any(
        not isinstance(task[k], str) or not re.fullmatch(r"[a-z][a-z0-9_-]{0,95}", task[k])
        for k in ("task_id", "record_id")
    ):
        raise ValueError("Invalid task/record ID")
    question = task["question"]
    if (
        not isinstance(question, str)
        or not 1 <= len(question.strip()) <= 500
        or any(ord(c) < 32 for c in question)
    ):
        raise ValueError("Question must be plain bounded text")



#### verify_payload

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
def verify_payload(payload, receipt_sha256):
    if canonical_sha256(payload) != receipt_sha256:
        raise ValueError("Tool receipt integrity mismatch")
    if set(payload) != {"values", "unit", "text", "citations", "artifact_refs"}:
        raise ValueError("Unexpected tool fields")
    if (
        not isinstance(payload["values"], list)
        or len(payload["values"]) > 3
        or any(
            isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v)
            for v in payload["values"]
        )
        or not payload["text"]
        or len(payload["text"]) > 5000
        or not payload["unit"]
        or not 1 <= len(payload["citations"]) <= 5
        or not 1 <= len(payload["artifact_refs"]) <= 20
    ):
        raise ValueError("Missing evidence or invalid numeric values")
    for citation in payload["citations"]:
        if not citation["source_url"].startswith("https://") or not re.fullmatch(
            r"[a-f0-9]{64}", citation["source_sha256"]
        ):
            raise ValueError("Invalid citation provenance")
    return True



#### Orchestrator

In [ ]:
%%inline_module src.ch06_multi_agent.core src/ch06_multi_agent/core.py both
class Orchestrator:
    """One request owns its DAG, 15-second deadline and at most eight tool attempts."""

    def __init__(self, tools, *, deadline_seconds=15.0):
        if not 0 < deadline_seconds <= 15:
            raise ValueError("Deadline outside locked budget")
        self.tools, self.deadline_seconds = tools, deadline_seconds

    def run(self, task, variant, taskset_sha256, *, fault=None):
        validate_task(task)
        if variant not in VARIANTS or not re.fullmatch(r"[a-f0-9]{64}", taskset_sha256):
            raise ValueError("Invalid variant/taskset identity")
        if fault not in {None, "transient", "permanent", "numeric", "citation", "timeout"}:
            raise ValueError("Undeclared internal fault")
        start, nodes, messages = time.perf_counter(), [], []
        deadline = start + self.deadline_seconds
        calls, peak = 0, 0
        values, citations, answer = [], [], "Không đủ bằng chứng để trả lời."
        status, verification = "abstained", "not_run"
        reason = policy_reason(task["question"], task["operation"])
        routed = task["domain_hint"] if variant == "A0" else route(task["question"])
        if routed != task["domain_hint"]:
            reason = reason or "routing_ambiguous_or_mismatch"

        def add(agent, tool, incoming, outgoing, outcome="ok", refs=None, elapsed=0):
            node_id = f"node_{len(nodes) + 1}"
            nodes.append(
                {
                    "node_id": node_id,
                    "agent": agent,
                    "parent_ids": [nodes[-1]["node_id"]] if nodes else [],
                    "tool": tool,
                    "input_sha256": canonical_sha256(incoming),
                    "output_sha256": canonical_sha256(outgoing),
                    "artifact_refs": refs or [],
                    "duration_ms": max(0, elapsed),
                    "outcome": outcome,
                }
            )
            messages.append({"node_id": node_id, "input": incoming, "output": outgoing})

        if variant != "A0":
            add("router", "none", task, {"route": routed, "policy": reason})
        tool = TOOLS.get((routed, task["operation"]), "none")
        reply = None
        role = "direct" if variant == "A0" else "specialist"
        if not reason:
            for attempt in range(2):
                seconds = deadline - time.perf_counter()
                if seconds <= 0:
                    status, reason = "timeout", "deadline"
                    break
                tick = time.perf_counter()
                reply = self.tools.execute(tool, task["record_id"], attempt, fault, seconds)
                calls += 1
                peak = max(peak, reply.get("peak_rss_bytes", 0))
                ok = reply["status"] == "ok"
                add(
                    role,
                    tool,
                    {"record_id": task["record_id"], "attempt": attempt},
                    reply,
                    "ok" if ok else ("timeout" if reply["status"] == "timeout" else "error"),
                    reply.get("payload", {}).get("artifact_refs", []),
                    (time.perf_counter() - tick) * 1000,
                )
                if ok:
                    p = reply["payload"]
                    values, citations, answer = p["values"], p["citations"], p["text"]
                    status = "succeeded"
                    break
                if reply["status"] == "timeout":
                    status, reason = "timeout", "deadline"
                    break
                if reply.get("code") != "transient" or attempt == 1:
                    status, reason = "failed", "tool_unavailable"
                    break
        else:
            add(role, "none", {"request_sha256": canonical_sha256(task)}, {"refusal": reason}, "abstain")
        if variant == "A2":
            tick = time.perf_counter()
            verification = "not_applicable"
            if status == "succeeded":
                try:
                    verify_payload(reply["payload"], reply["receipt_sha256"])
                    verification = "passed"
                except (ValueError, KeyError, TypeError):
                    status, reason, verification = "abstained", "invalid_tool_evidence", "rejected"
                    values, citations, answer = [], [], "Bằng chứng công cụ không hợp lệ; từ chối suy diễn."
            add(
                "verifier",
                "none",
                {"receipt": reply},
                {"verification": verification, "reason": reason},
                "abstain" if verification == "rejected" else "ok",
                elapsed=(time.perf_counter() - tick) * 1000,
            )
            add(
                "reporter",
                "none",
                {"status": status, "values": values, "citations": citations},
                {"answer": answer, "warning": WARNING},
            )
        if time.perf_counter() > deadline:
            status, reason, values, citations = "timeout", "deadline", [], []
        if status != "succeeded":
            answer = f"Không thực hiện yêu cầu: {reason or 'insufficient_evidence'}."
        trace = {
            "schema_version": "3.0.0",
            "evidence_kind": "measured",
            "trace_id": f"trace_{canonical_sha256(task)[:20]}_{variant.lower()}",
            "task_id": task["task_id"],
            "task_set_sha256": taskset_sha256,
            "variant": variant,
            "domain": task["domain_hint"],
            "status": status,
            "nodes": nodes,
            "answer": answer,
            "citations": citations,
            "limitations": [WARNING],
            "latency_ms": (time.perf_counter() - start) * 1000,
        }
        validate_document("agent", trace, publish=True)
        return {
            "trace": trace,
            "messages": messages,
            "values": values,
            "route": routed,
            "verification": verification,
            "reason": reason,
            "tool_calls": calls,
            "peak_child_rss_bytes": peak,
            "message_count": len(nodes),
        }



### `src.ch06_multi_agent.benchmark`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch06_multi_agent.benchmark src/ch06_multi_agent/benchmark.py both
"""Independent saved-output rubric, grouped task audit and irreversible claims."""

import json

import math

from collections import Counter

from src.ch06_multi_agent.core import DOMAINS, validate_task



#### claim

In [ ]:
%%inline_module src.ch06_multi_agent.benchmark src/ch06_multi_agent/benchmark.py both
def claim(path, document):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8", newline="\n") as stream:
        json.dump(document, stream, ensure_ascii=False, sort_keys=True, allow_nan=False)
        stream.write("\n")



#### audit_groups

In [ ]:
%%inline_module src.ch06_multi_agent.benchmark src/ch06_multi_agent/benchmark.py both
def audit_groups(rows, *, require_counts=True):
    ids, groups, sources = set(), {}, {}
    for row in rows:
        validate_task(row["request"])
        key = row["request"]["task_id"]
        if key in ids or row["domain"] != row["request"]["domain_hint"]:
            raise ValueError("Duplicate task or domain mismatch")
        ids.add(key)
        for values, field in [(groups, "group_id"), (sources, "source_group")]:
            group = (row["domain"], row[field])
            if group in values and values[group] != row["split"]:
                raise ValueError("Source/template group crosses splits")
            values[group] = row["split"]
    if require_counts:
        counts = Counter((r["domain"], r["split"], r["category"]) for r in rows)
        for domain in DOMAINS:
            for split, count in [("train", 12), ("dev", 4), ("test", 4)]:
                for category in ("numeric", "evidence", "ood", "unsafe", "tool_failure"):
                    if counts[(domain, split, category)] != count:
                        raise ValueError("Task matrix must be 100/domain with exact grouped60/20/20")
        if len(rows) != 300:
            raise ValueError("Unexpected task count")
    return {"tasks": len(rows), "template_groups": len(groups), "source_groups": len(sources), "overlap": 0}



#### _numeric

In [ ]:
%%inline_module src.ch06_multi_agent.benchmark src/ch06_multi_agent/benchmark.py both
def _numeric(actual, expected):
    return (
        isinstance(actual, list)
        and len(actual) == len(expected)
        and all(
            isinstance(a, (int, float))
            and not isinstance(a, bool)
            and math.isfinite(a)
            and math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-8)
            for a, b in zip(actual, expected, strict=True)
        )
    )



#### score

In [ ]:
%%inline_module src.ch06_multi_agent.benchmark src/ch06_multi_agent/benchmark.py both
def score(row, output):
    trace, gold = output["trace"], row["gold"]
    succeeded = trace["status"] == "succeeded"
    wants_answer = gold["status"] == "succeeded"
    expected = gold.get("payload")
    numeric = None
    citation = None
    if expected is not None:
        numeric = (
            int(succeeded and _numeric(output["values"], expected["values"])) if expected["values"] else None
        )
        citation = int(succeeded and trace["citations"] == expected["citations"])
    success = (
        (succeeded and (numeric in {None, 1}) and citation == 1 and trace["answer"] == expected["text"])
        if wants_answer
        else (trace["status"] == gold["status"] and not output["values"] and not trace["citations"])
    )
    wrong_answer = succeeded and (not wants_answer or numeric == 0 or citation == 0)
    return {
        "task_success": int(success),
        "answer_completion": int(succeeded),
        "routing_accuracy": None if trace["variant"] == "A0" else int(output["route"] == row["domain"]),
        "numeric_parity": numeric,
        "citation_correctness": citation,
        "unsupported_claim": int(wrong_answer),
        "unsafe_claim": int(row["category"] == "unsafe" and succeeded),
        "refusal_correctness": int(trace["status"] == "abstained") if gold["status"] == "abstained" else None,
        "latency_ms": trace["latency_ms"],
        "message_count": output["message_count"],
        "tool_calls": output["tool_calls"],
        "peak_child_rss_bytes": output["peak_child_rss_bytes"],
    }



### `src.ch06_multi_agent.executor`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch06_multi_agent.executor src/ch06_multi_agent/executor.py both
"""Local CPU worker isolation with hard tool timeout; no arbitrary process commands."""

import multiprocessing as mp

import resource

import threading

from src.common.contracts import canonical_sha256



#### _child

In [ ]:
%%inline_module src.ch06_multi_agent.executor src/ch06_multi_agent/executor.py both
def _child(connection, function, tool, record, attempt, fault):
    try:
        value = function(tool, record, attempt, fault)
        if isinstance(value, dict) and value.get("status") == "error":
            reply = value
        else:
            receipt = canonical_sha256(value)
            if fault == "numeric":
                value["values"][0] += 1.0
            elif fault == "citation":
                value["citations"] = []
            reply = {"status": "ok", "payload": value, "receipt_sha256": receipt}
        reply["peak_rss_bytes"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
        connection.send(reply)
    except (OSError, ValueError, TypeError, KeyError, RuntimeError, ArithmeticError, ImportError):
        connection.send({"status": "error", "code": "tool_error", "peak_rss_bytes": 0})
    finally:
        connection.close()



#### ProcessExecutor

In [ ]:
%%inline_module src.ch06_multi_agent.executor src/ch06_multi_agent/executor.py both
class ProcessExecutor:
    """WSL/Linux single-threaded driver only; not a web-server fork strategy."""

    def __init__(self, function):
        self.function, self.active = function, False

    def execute(self, tool, record_id, attempt, fault, seconds):
        if self.active or threading.active_count() != 1:
            raise ValueError("One single-threaded tool worker at a time")
        if seconds <= 0:
            return {"status": "timeout", "code": "deadline", "peak_rss_bytes": 0}
        context = mp.get_context("fork")
        reader, writer = context.Pipe(duplex=False)
        worker = context.Process(target=_child, args=(writer, self.function, tool, record_id, attempt, fault))
        self.active = True
        try:
            worker.start()
            writer.close()
            if reader.poll(seconds):
                try:
                    return reader.recv()
                except EOFError:
                    return {"status": "error", "code": "worker_exit", "peak_rss_bytes": 0}
            return {"status": "timeout", "code": "deadline", "peak_rss_bytes": 0}
        finally:
            if worker.pid is not None:
                if worker.is_alive():
                    worker.terminate()
                worker.join(timeout=0.2)
                if worker.is_alive():
                    worker.kill()
                    worker.join()
            reader.close()
            writer.close()
            self.active = False



### `src.ch06_multi_agent.service`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch06_multi_agent.service src/ch06_multi_agent/service.py both
"""Versioned deterministic function contracts and local V3 request adapter."""

import re

from src.ch06_multi_agent.core import DOMAINS, VARIANTS, Orchestrator, validate_task

from src.common.contracts import canonical_sha256, validate_document

FUNCTIONS = {
    "validate": (
        "validate_task(task)",
        "Kiểm tra đầu vào",
        "JSON5 trường",
        "task hợp lệ",
        "Sai ID/độ dài/trường lạ: từ chối",
    ),
    "policy": (
        "policy_reason(question, operation)",
        "Áp dụng an toàn chung",
        "question + operation",
        "reason hoặc null",
        "Rule hữu hạn, không chứng nhận an toàn toàn diện",
    ),
    "route": (
        "route(question)",
        "Chọn một miền",
        "text tối đa500 ký tự",
        "diabetes/house/gold hoặc null",
        "Không biết hoặc trộn miền: abstain",
    ),
    "direct_tool": (
        "ToolCatalog.call(tool, record_id)",
        "Gọi tool theo miền đã chọn",
        "allowlisted record",
        "payload + provenance",
        "Không chạy shell/URL/path từ câu hỏi",
    ),
    "specialist": (
        "ProcessExecutor.execute(...)",
        "Thực hiện tool trong process có hạn giờ",
        "tool + record + deadline",
        "receipt hoặc lỗi",
        "Retry transient1 lần; terminate/join khi timeout",
    ),
    "verify": (
        "verify_payload(payload, receipt_sha256)",
        "Kiểm chứng bằng chứng",
        "payload + receipt hash",
        "pass hoặc từ chối",
        "Kiểm integrity/finite/citation, không xác nhận chuyên môn",
    ),
    "report": (
        "Orchestrator.run(...): reporter",
        "Tổng hợp phản hồi có giới hạn",
        "status + values + citations",
        "answer + cảnh báo",
        "Không tự thêm lời khuyên hoặc con số",
    ),
    "emit": (
        "validate_document('agent', trace)",
        "Đóng gói trace V3",
        "DAG + action messages",
        "agent.schema3.0.0",
        "Không ghi chain-of-thought; DAG phải topo",
    ),
}

COMPOSITIONS = {
    "A0": ["validate", "policy", "direct_tool", "emit"],
    "A1": ["validate", "policy", "route", "specialist", "emit"],
    "A2": ["validate", "policy", "route", "specialist", "verify", "report", "emit"],
}



#### pipeline_contract

In [ ]:
%%inline_module src.ch06_multi_agent.service src/ch06_multi_agent/service.py both
def pipeline_contract(domain, variant):
    if domain not in DOMAINS or variant not in VARIANTS:
        raise ValueError("Unknown pipeline")
    return {
        "schema_version": "phase31.agent_pipeline.1",
        "domain": domain,
        "variant": variant,
        "parameter_count": None,
        "learnable_layers": None,
        "composition": COMPOSITIONS[variant],
        "composition_plain": " → ".join(FUNCTIONS[k][1] for k in COMPOSITIONS[variant]),
        "functions": [
            {
                "id": key,
                **dict(zip(("signature", "role", "input", "output", "corner_case"), FUNCTIONS[key])),
                "source": "src/ch06_multi_agent/"
                + (
                    "executor.py"
                    if key == "specialist"
                    else ("tools.py" if key == "direct_tool" else "core.py")
                ),
                "formula": None,
                "learned_parameters": None,
            }
            for key in COMPOSITIONS[variant]
        ],
        "budget": {"deadline_seconds": 15, "max_tool_calls": 8, "retry_per_tool": 1},
        "actual_backend": "numpy_cpu_deterministic",
        "api_key_required": False,
    }



#### sample_request

In [ ]:
%%inline_module src.ch06_multi_agent.service src/ch06_multi_agent/service.py both
def sample_request(domain):
    if domain not in DOMAINS:
        raise ValueError("Unknown domain")
    topic = {"diabetes": "tiểu đường", "house": "giá nhà", "gold": "giá vàng"}[domain]
    return {
        "schema_version": "3.0.0",
        "dataset_id": f"ch6_{domain}_tasks",
        "question": f"{topic}: xem kết quả học thuật record={domain}_numeric_000",
        "variants": list(VARIANTS),
    }



#### decode_question

In [ ]:
%%inline_module src.ch06_multi_agent.service src/ch06_multi_agent/service.py both
def decode_question(domain, question):
    if domain not in DOMAINS or not isinstance(question, str) or len(question) > 500:
        raise ValueError("Unsupported local request")
    matches = re.findall(r"record=([^\s;,]+)", question)
    operation, key = "unsupported", "no_record"
    if matches:
        if len(matches) != 1 or not re.fullmatch(
            r"(?:diabetes|house|gold)_(?:numeric|evidence)_\d{3}", matches[0]
        ):
            raise ValueError("Require exactly one approved record ID")
        key = matches[0]
        if not key.startswith(domain + "_"):
            raise ValueError("Record/domain mismatch")
        operation = "numeric" if "_numeric_" in key else "evidence"
    task = {
        "task_id": "demo_" + canonical_sha256(question)[:16],
        "domain_hint": domain,
        "question": question,
        "record_id": key,
        "operation": operation,
    }
    validate_task(task)
    return task



#### AgentService

In [ ]:
%%inline_module src.ch06_multi_agent.service src/ch06_multi_agent/service.py both
class AgentService:
    def __init__(self, tools, taskset_sha256):
        self.engine, self.taskset_sha256 = Orchestrator(tools), taskset_sha256

    def compare(self, request):
        validate_document("api#agent_task", request)
        dataset = request["dataset_id"]
        if dataset not in {f"ch6_{d}_tasks" for d in DOMAINS}:
            raise ValueError("Unknown agent dataset")
        domain = dataset.removeprefix("ch6_").removesuffix("_tasks")
        task = decode_question(domain, request["question"])
        return {
            "scope": "local_task_demo_not_http",
            "dataset_id": dataset,
            "outputs": {v: self.engine.run(task, v, self.taskset_sha256) for v in VARIANTS},
            "historical_test_evaluation_executed": False,
            "limitation": "Structured record-bound demo, not arbitrary natural-language reasoning or a clinical/investment assistant",
        }



### `src.common.model_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
"""One architecture plan supplies all backends, contracts and explanations.

Canonical activations: NHWC images, NTF sequences, dense IO, conv HWIO,
recurrent (input, gates*hidden)/(hidden, gates*hidden), LSTM i/f/g/o.
"""

from __future__ import annotations

import copy

import numpy as np

from src.common.contracts import PROTOCOL, canonical_sha256, read_json, validate_document

BACKENDS = {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}

LEARNED = {"dense", "conv2d", "rnn", "lstm"}



#### build_plan

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def build_plan(family, input_shape, outputs, *, tiny=False):
    protocol = read_json(PROTOCOL)
    if (
        family not in protocol["families"]
        or not input_shape
        or outputs < 1
        or any(not isinstance(v, int) or v < 1 for v in input_shape)
    ):
        raise ValueError("Invalid family/input/output dimensions")
    if len(input_shape) != {"mlp": 1, "cnn": 3, "rnn": 2, "lstm": 2}[family]:
        raise ValueError("Input rank mismatch")
    shape = list(input_shape)
    steps = []

    def add(kind, output, **options):
        nonlocal shape
        step = {
            "id": f"{kind}_{len(steps) + 1}",
            "kind": kind,
            "input_shape": shape,
            "output_shape": list(output),
            **options,
        }
        steps.append(step)
        shape = list(output)

    policy = protocol["families"][family]
    if family == "mlp":
        for units in [2] * 5 if tiny else policy["hidden_units"]:
            add("dense", [units])
            add("activation", shape, activation="relu")
    elif family == "cnn":
        for i, channels in enumerate([2] * 5 if tiny else policy["channels"]):
            add("conv2d", [*shape[:2], channels], kernel_size=3, padding="same")
            add("activation", shape, activation="relu")
            if i in policy["pool_after_convolutions"]:
                if min(shape[:2]) < 2:
                    raise ValueError("Image too small for two pooling stages")
                add("pooling", [shape[0] // 2, shape[1] // 2, shape[2]], pool="max2")
        add("pooling", [shape[-1]], pool="global_average")
        add("dense", [2 if tiny else policy["dense_hidden"]])
        add("activation", shape, activation="relu")
    else:
        for i, units in enumerate([2] * 3 if tiny else policy["hidden_units"]):
            add(family, [shape[0], units] if i < 2 else [units], return_sequences=i < 2)
        for units in [2, 2] if tiny else policy["dense_hidden"]:
            add("dense", [units])
            add("activation", shape, activation="relu")
    add("dense", [outputs])
    return {
        "plan_version": "phase25.1",
        "family": family,
        "input_shape": list(input_shape),
        "output_shape": shape,
        "steps": steps,
        "tiny_synthetic_only": tiny,
    }



#### parameter_shapes

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def parameter_shapes(step):
    kind = step["kind"]
    width = step["input_shape"][-1]
    units = step["output_shape"][-1]
    if kind == "dense":
        return {"w": (width, units), "b": (units,)}
    if kind == "conv2d":
        return {"w": (step["kernel_size"], step["kernel_size"], width, units), "b": (units,)}
    if kind in {"rnn", "lstm"}:
        gates = 4 if kind == "lstm" else 1
        return {"w": (width, gates * units), "u": (units, gates * units), "b": (gates * units,)}
    return {}



#### canonical_weights

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def canonical_weights(plan, seed, dtype="float32"):
    rng = np.random.default_rng(seed)
    result = {}
    for step in plan["steps"]:
        for name, shape in parameter_shapes(step).items():
            value = (
                np.zeros(shape, dtype=dtype)
                if name == "b"
                else rng.normal(0, np.sqrt(1 / np.prod(shape[:-1])), shape).astype(dtype)
            )
            if step["kind"] == "lstm" and name == "b":
                units = step["output_shape"][-1]
                value[units : 2 * units] = 1
            result[f"{step['id']}/{name}"] = value
    return result



#### formula_for

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def formula_for(step):
    kind = step["kind"]
    expressions = {
        "dense": "y = xW + b",
        "conv2d": "y = conv_same(x, W) + b",
        "rnn": "h_t = tanh(x_t W + h_previous U + b)",
        "lstm": "(i,f,g,o) = (sigmoid,sigmoid,tanh,sigmoid)(x_t W + h_previous U + b); c_t = f*c_previous + i*g; h_t = o*tanh(c_t)",
    }
    if kind not in expressions:
        return None
    meanings = {
        "x": "tensor đầu vào (x_t: tại bước t)",
        "y": "tensor đầu ra",
        "W": "trọng số đầu vào",
        "b": "bias học được",
    }
    if kind in {"rnn", "lstm"}:
        meanings.update(
            {
                "h_t": "trạng thái ở bước t",
                "h_previous": "trạng thái bước trước, bắt đầu bằng 0",
                "U": "trọng số trạng thái",
                "t": "chỉ số bước thời gian",
            }
        )
    if kind == "rnn":
        meanings["tanh"] = "hàm ép giá trị vào khoảng -1 đến 1"
    if kind == "conv2d":
        meanings["conv_same"] = "trượt bộ lọc 3×3, đệm 0 để giữ kích thước ảnh"
    if kind == "lstm":
        meanings.update(
            {
                "i,f,o": "cổng nhận, quên, xuất trong 0..1",
                "g": "ứng viên bộ nhớ",
                "c_t": "bộ nhớ mới",
                "c_previous": "bộ nhớ bước trước, bắt đầu bằng 0",
                "sigmoid": "hàm ép vào khoảng 0 đến 1",
                "tanh": "hàm ép vào khoảng -1 đến 1",
                "*": "nhân từng phần tử",
            }
        )
    return {
        "expression": expressions[kind],
        "symbols": [{"symbol": k, "meaning": v} for k, v in meanings.items()],
        "why_needed": "Đối chiếu phép tính forward trong code scratch.",
        "plain_explanation": {
            "dense": "Kết hợp các giá trị đầu vào bằng trọng số rồi cộng bias.",
            "conv2d": "Tìm đặc trưng cục bộ bằng bộ lọc trượt trên ảnh.",
            "rnn": "Kết hợp dữ liệu hiện tại và thông tin nhớ từ bước trước.",
            "lstm": "Chọn thông tin cần giữ, bổ sung và xuất qua các cổng.",
        }[kind],
        "example": f"Shape {step['input_shape']} → {step['output_shape']} cho một mẫu; batch được giữ nguyên.",
    }



#### make_contract

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def make_contract(plan, implementation, evidence_kind, provenance=None):
    if implementation not in BACKENDS:
        raise ValueError("Unknown implementation")
    if evidence_kind != "synthetic_fixture" and provenance is None:
        raise ValueError("Measured contracts require provenance")
    if plan["tiny_synthetic_only"] and evidence_kind != "synthetic_fixture":
        raise ValueError("Tiny plan is not a production model")
    family = plan["family"]
    dataset = {
        "mlp": "ch2_cdc_diabetes",
        "cnn": "ch3_cifar10",
        "rnn": "ch4_gold_pink_sheet",
        "lstm": "ch4_gold_pink_sheet",
    }[family]
    digest = canonical_sha256(plan)
    provenance = provenance or {
        "dataset_id": dataset,
        "config_sha256": digest,
        "dataset_sha256": digest,
        "comparison_group_sha256": digest,
        "preprocessing": {
            "path": "artifacts/manifests/phase25/synthetic_preprocessing.json",
            "sha256": digest,
        },
    }
    layers = []
    functions = []
    keras_names = {
        "dense": "Dense",
        "conv2d": "Conv2D",
        "rnn": "SimpleRNN",
        "lstm": "LSTM",
        "activation": "ReLU",
        "pooling": "MaxPooling2D/GlobalAveragePooling2D",
    }
    torch_names = {
        "dense": "Linear",
        "conv2d": "Conv2d",
        "rnn": "RNN",
        "lstm": "LSTM",
        "activation": "ReLU",
        "pooling": "MaxPool2d/AdaptiveAvgPool2d",
    }
    roles = {
        "dense": "Kết hợp các đặc trưng",
        "conv2d": "Tìm đặc trưng trong vùng ảnh",
        "rnn": "Nhớ thông tin theo thời gian",
        "lstm": "Giữ/quên thông tin bằng bộ nhớ và cổng",
        "activation": "Giữ tín hiệu dương, tạo khả năng học phi tuyến",
        "pooling": "Rút gọn vùng ảnh",
    }
    for step in plan["steps"]:
        count = int(sum(np.prod(shape) for shape in parameter_shapes(step).values()))
        inp = ["batch", *step["input_shape"]]
        out = ["batch", *step["output_shape"]]
        layers.append(
            {
                "id": step["id"],
                "kind": step["kind"],
                "parameter_count": count,
                "input_shape": inp,
                "output_shape": out,
                "function_id": step["id"],
            }
        )
        functions.append(
            {
                "id": step["id"],
                "signature": f"{step['kind']}(x, weights) -> y",
                "role": roles[step["kind"]],
                "input_shape": inp,
                "output_shape": out,
                "parameters": count,
                "corner_case": "Sai rank/shape bị từ chối; trạng thái RNN/LSTM reset giữa các window; không bỏ batch cuối.",
                "formula": formula_for(step),
                "mapping": {
                    "scratch": f"src.common.numpy_engine.NumpyNetwork.forward[{step['kind']}]",
                    "keras": f"keras.layers.{keras_names[step['kind']]}",
                    "pytorch": f"torch.nn.{torch_names[step['kind']]}",
                },
            }
        )
    contract = {
        "schema_version": "3.0.0",
        "evidence_kind": evidence_kind,
        "model_id": f"{provenance['dataset_id']}_{family}_{implementation}",
        "dataset_id": provenance["dataset_id"],
        "family": family,
        "implementation": implementation,
        "backend": BACKENDS[implementation],
        "parameter_count": sum(x["parameter_count"] for x in layers),
        "input_shape": ["batch", *plan["input_shape"]],
        "output_shape": ["batch", *plan["output_shape"]],
        "layers": layers,
        "functions": functions,
        "composition": [x["id"] for x in functions],
        "composition_plain": "đầu vào → "
        + " → ".join(f"{roles[x['kind']]} {x['output_shape']}" for x in plan["steps"])
        + " → đầu ra (logit/giá trị chuẩn hóa)",
        "recurrent_unroll": "Ở mỗi tầng l=1..3, lặp t=1..T theo thứ tự; khởi tạo h (và c) bằng 0 cho mỗi window. Tầng 1–2 trả toàn chuỗi, tầng 3 trả bước cuối → 3 Dense."
        if family in {"rnn", "lstm"}
        else None,
        **{
            key: provenance[key]
            for key in ("config_sha256", "dataset_sha256", "comparison_group_sha256", "preprocessing")
        },
    }
    validate_model(contract)
    return contract



#### validate_model

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def validate_model(contract):
    validate_document("model", contract)
    functions = {row["id"]: row for row in contract["functions"]}
    if contract["composition"] != [row["function_id"] for row in contract["layers"]]:
        raise ValueError("Composition order differs from execution")
    previous = contract["input_shape"]
    for layer in contract["layers"]:
        function = functions[layer["function_id"]]
        if (
            layer["input_shape"] != previous
            or any(function[k] != layer[k] for k in ("input_shape", "output_shape"))
            or function["parameters"] != layer["parameter_count"]
        ):
            raise ValueError("Shape/function/parameter trace mismatch")
        kind = layer["kind"]
        width = layer["input_shape"][-1]
        units = layer["output_shape"][-1]
        expected = 0
        if kind == "dense":
            expected = (width + 1) * units
        elif kind == "conv2d":
            expected = (9 * width + 1) * units
        elif kind in {"rnn", "lstm"}:
            expected = (width + units + 1) * units * (4 if kind == "lstm" else 1)
        if layer["parameter_count"] != expected:
            raise ValueError("Parameter count does not match tensor dimensions")
        previous = layer["output_shape"]
    if previous != contract["output_shape"]:
        raise ValueError("Final output shape mismatch")



#### render_functions

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def render_functions(contract):
    validate_model(contract)
    functions = copy.deepcopy(contract["functions"])
    technical = "F = " + " ∘ ".join(reversed(contract["composition"]))
    table = "| Hàm | Vai trò | Input → Output | Tham số |\n|---|---|---|---:|\n"
    table += "".join(
        f"| {f['id']} | {f['role']} | {f['input_shape']} → {f['output_shape']} | {f['parameters']} |\n"
        for f in functions
    )
    markdown = f"# {contract['model_id']}\n\n{contract['composition_plain']}\n\n`{technical}`\n\n{contract['recurrent_unroll'] or ''}\n\n{table}"
    for function in functions:
        markdown += f"\n## {function['id']}\n\n`{function['signature']}` — {function['role']}.\n\n{function['corner_case']}\n\nMapping: {function['mapping']}\n"
        formula = function["formula"]
        if formula:
            markdown += (
                f"\n{formula['plain_explanation']}\n\n`{formula['expression']}`\n\n"
                + "; ".join(f"{s['symbol']}: {s['meaning']}" for s in formula["symbols"])
                + f".\n\n{formula['example']}\n"
            )
    return {
        "technical": technical,
        "plain": contract["composition_plain"],
        "functions": functions,
        "markdown": markdown,
    }



#### ModelRegistry

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
class ModelRegistry:
    def __init__(self):
        self._models = {}

    def register(self, contract):
        validate_model(contract)
        if contract["model_id"] in self._models:
            raise ValueError("Duplicate model ID")
        self._models[contract["model_id"]] = copy.deepcopy(contract)

    def get(self, model_id):
        return copy.deepcopy(self._models[model_id])



### `src.common.numpy_engine`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
"""Educational NumPy forward/backward primitives, without autograd/frameworks.

This reference engine is first verified on synthetic data in Phase25. Real
dataset training/evaluation is implemented and accepted in Phases26–29.
"""

from __future__ import annotations

import numpy as np

from src.common.model_core import parameter_shapes



#### sigmoid

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def sigmoid(value):
    # Equivalent stable expression, without clipping the mathematical derivative.
    return np.exp(-np.logaddexp(0, -value))



#### loss_and_gradient

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def loss_and_gradient(logits, target, task, sample_weight=None):
    logits = np.asarray(logits)
    target = np.asarray(target)
    if logits.ndim != 2 or len(logits) == 0 or not np.isfinite(logits).all() or not np.isfinite(target).all():
        raise ValueError("Loss requires finite, nonempty [batch, outputs] logits")
    if task == "multiclass":
        if (
            target.shape != (len(logits),)
            or not np.equal(target, np.floor(target)).all()
            or np.any(target < 0)
            or np.any(target >= logits.shape[1])
        ):
            raise ValueError("Invalid class IDs")
        labels = target.astype(int)
        shift = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(shift)
        probs /= probs.sum(axis=1, keepdims=True)
        rows = np.log(np.exp(shift).sum(axis=1)) - shift[np.arange(len(logits)), labels]
        gradient = probs.copy()
        gradient[np.arange(len(logits)), labels] -= 1
    elif task in {"mse", "binary"}:
        if target.shape != logits.shape:
            raise ValueError("Target shape mismatch")
        if task == "binary":
            if not np.isin(target, [0, 1]).all():
                raise ValueError("Binary labels must be zero or one")
            rows = (np.logaddexp(0, logits) - target * logits).mean(axis=1)
            gradient = (sigmoid(logits) - target) / logits.shape[1]
        else:
            rows = np.mean((logits - target) ** 2, axis=1)
            gradient = 2 * (logits - target) / logits.shape[1]
    else:
        raise ValueError("Unknown loss")
    weights = np.ones(len(logits)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    if (
        weights.shape != (len(logits),)
        or not np.isfinite(weights).all()
        or np.any(weights < 0)
        or weights.sum() <= 0
    ):
        raise ValueError("Invalid sample weights")
    normalized = weights / weights.sum()
    return float(rows @ normalized), (gradient * normalized[:, None]).astype(logits.dtype)



#### numerical_gradient

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def numerical_gradient(function, value, epsilon=1e-5):
    if value.dtype != np.float64 or epsilon <= 0:
        raise ValueError("Numerical checks require float64 and positive epsilon")
    gradient = np.zeros_like(value)
    for index in np.ndindex(value.shape):
        previous = value[index]
        try:
            value[index] = previous + epsilon
            positive = function()
            value[index] = previous - epsilon
            negative = function()
        finally:
            value[index] = previous
        gradient[index] = (positive - negative) / (2 * epsilon)
    return gradient



#### NumpyNetwork

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
class NumpyNetwork:
    def __init__(self, plan, weights):
        self.plan = plan
        expected = {
            f"{s['id']}/{k}": shape for s in plan["steps"] for k, shape in parameter_shapes(s).items()
        }
        if set(expected) != set(weights) or any(
            weights[k].shape != shape or not np.isfinite(weights[k]).all() for k, shape in expected.items()
        ):
            raise ValueError("Canonical weight names/shapes/values mismatch")
        self.weights = {key: value.copy() for key, value in weights.items()}
        self.cache = []

    def forward(self, x, *, trace=False):
        x = np.asarray(x, dtype=next(iter(self.weights.values())).dtype)
        if (
            x.ndim != len(self.plan["input_shape"]) + 1
            or list(x.shape[1:]) != self.plan["input_shape"]
            or len(x) == 0
            or not np.isfinite(x).all()
        ):
            raise ValueError("Input tensor shape/value mismatch")
        self.cache = []
        traces = []
        for step in self.plan["steps"]:
            before = x
            kind = step["kind"]
            prefix = step["id"]
            params = {key: self.weights[f"{prefix}/{key}"] for key in parameter_shapes(step)}
            extra = None
            if kind == "dense":
                x = before @ params["w"] + params["b"]
            elif kind == "activation":
                x = np.maximum(before, 0)
            elif kind == "conv2d":
                padded = np.pad(before, ((0, 0), (1, 1), (1, 1), (0, 0)))
                x = np.zeros((len(before), *step["output_shape"]), dtype=before.dtype)
                for i in range(3):
                    for j in range(3):
                        x += np.einsum(
                            "nhwc,co->nhwo",
                            padded[:, i : i + before.shape[1], j : j + before.shape[2]],
                            params["w"][i, j],
                        )
                x += params["b"]
                extra = padded
            elif kind == "pooling":
                if step["pool"] == "global_average":
                    x = before.mean(axis=(1, 2))
                else:
                    n, h, w, c = before.shape
                    hh, ww = h // 2, w // 2
                    patches = (
                        before[:, : hh * 2, : ww * 2]
                        .reshape(n, hh, 2, ww, 2, c)
                        .transpose(0, 1, 3, 5, 2, 4)
                        .reshape(n, hh, ww, c, 4)
                    )
                    extra = patches.argmax(axis=-1)
                    x = patches.max(axis=-1)
            else:
                units = step["output_shape"][-1]
                h = np.zeros((len(before), units), dtype=before.dtype)
                c = np.zeros_like(h)
                states = []
                outputs = []
                for t in range(before.shape[1]):
                    z = before[:, t] @ params["w"] + h @ params["u"] + params["b"]
                    if kind == "rnn":
                        next_h = np.tanh(z)
                        states.append((before[:, t], h, next_h))
                        h = next_h
                    else:
                        zi, zf, zg, zo = np.split(z, 4, axis=1)
                        i, f, g, o = sigmoid(zi), sigmoid(zf), np.tanh(zg), sigmoid(zo)
                        next_c = f * c + i * g
                        next_h = o * np.tanh(next_c)
                        states.append((before[:, t], h, c, i, f, g, o, next_c))
                        h, c = next_h, next_c
                    outputs.append(h)
                full = np.stack(outputs, axis=1)
                x = full if step["return_sequences"] else full[:, -1]
                extra = states
            self.cache.append((step, before, params, extra))
            traces.append({"id": prefix, "input_shape": list(before.shape), "output_shape": list(x.shape)})
        return (x, traces) if trace else x

    def backward(self, gradient):
        gradients = {}
        for step, before, params, extra in reversed(self.cache):
            kind = step["kind"]
            prefix = step["id"]
            grads = {key: np.zeros_like(value) for key, value in params.items()}
            if kind == "dense":
                grads["w"] = before.T @ gradient
                grads["b"] = gradient.sum(axis=0)
                gradient = gradient @ params["w"].T
            elif kind == "activation":
                gradient = gradient * (before > 0)
            elif kind == "conv2d":
                grads["b"] = gradient.sum(axis=(0, 1, 2))
                padded = extra
                dx = np.zeros_like(padded)
                for i in range(3):
                    for j in range(3):
                        patch = padded[:, i : i + before.shape[1], j : j + before.shape[2]]
                        grads["w"][i, j] = np.einsum("nhwc,nhwo->co", patch, gradient)
                        dx[:, i : i + before.shape[1], j : j + before.shape[2]] += np.einsum(
                            "nhwo,co->nhwc", gradient, params["w"][i, j]
                        )
                gradient = dx[:, 1:-1, 1:-1]
            elif kind == "pooling":
                if step["pool"] == "global_average":
                    gradient = np.broadcast_to(
                        gradient[:, None, None, :] / (before.shape[1] * before.shape[2]), before.shape
                    ).copy()
                else:
                    n, h, w, c = before.shape
                    hh, ww = h // 2, w // 2
                    patches = np.zeros((n, hh, ww, c, 4), dtype=before.dtype)
                    np.put_along_axis(patches, extra[..., None], gradient[..., None], axis=-1)
                    dx = np.zeros_like(before)
                    dx[:, : hh * 2, : ww * 2] = (
                        patches.reshape(n, hh, ww, c, 2, 2)
                        .transpose(0, 1, 4, 2, 5, 3)
                        .reshape(n, hh * 2, ww * 2, c)
                    )
                    gradient = dx
            else:
                full_gradient = np.zeros(
                    (len(before), before.shape[1], step["output_shape"][-1]), dtype=before.dtype
                )
                if step["return_sequences"]:
                    full_gradient += gradient
                else:
                    full_gradient[:, -1] = gradient
                dh = np.zeros_like(full_gradient[:, 0])
                dc = np.zeros_like(dh)
                dx = np.zeros_like(before)
                for t in reversed(range(before.shape[1])):
                    dh = dh + full_gradient[:, t]
                    if kind == "rnn":
                        xt, previous_h, h = extra[t]
                        dz = dh * (1 - h * h)
                    else:
                        xt, previous_h, previous_c, i, f, g, o, c = extra[t]
                        tanhc = np.tanh(c)
                        dc = dc + dh * o * (1 - tanhc * tanhc)
                        dz = np.concatenate(
                            (
                                dc * g * i * (1 - i),
                                dc * previous_c * f * (1 - f),
                                dc * i * (1 - g * g),
                                dh * tanhc * o * (1 - o),
                            ),
                            axis=1,
                        )
                        dc = dc * f
                    grads["w"] += xt.T @ dz
                    grads["u"] += previous_h.T @ dz
                    grads["b"] += dz.sum(axis=0)
                    dx[:, t] = dz @ params["w"].T
                    dh = dz @ params["u"].T
                gradient = dx
            gradients.update({f"{prefix}/{name}": value for name, value in grads.items()})
        return gradients, gradient

    def loss_gradients(self, x, y, task, sample_weight=None):
        logits = self.forward(x)
        loss, gradient = loss_and_gradient(logits, y, task, sample_weight)
        gradients, dx = self.backward(gradient)
        return loss, gradients, dx



### `src.common.tensor_adapters`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
"""Canonical Phase25 tensors. Apply frozen train statistics; never fit here.

Order is numeric input columns followed by categorical input columns/vocabulary.
The ordered names are persisted explicitly, independently of JSON object order.
"""

from __future__ import annotations

import copy

import numpy as np

import pandas as pd



#### TensorAdapter

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
class TensorAdapter:
    def __init__(self, state):
        self.state = copy.deepcopy(state)
        if "target_transform" in self.state and "name" not in self.state["target_transform"]:
            self.state["target_transform"]["name"] = "standardize"
        state = self.state
        names = state["input_columns"]
        numeric, categorical = state["numeric"], state["categorical"]
        if (
            state.get("fit_scope") != "train_only"
            or len(set(names)) != len(names)
            or set(numeric) & set(categorical)
            or set(names) != set(numeric) | set(categorical)
        ):
            raise ValueError("Invalid train-only feature schema")
        for rule in numeric.values():
            if not all(np.isfinite(rule[k]) for k in ("median", "mean", "std")) or rule["std"] <= 0:
                raise ValueError("Invalid frozen numeric statistics")
        for values in categorical.values():
            if not values or len(set(values)) != len(values) or not all(isinstance(v, str) for v in values):
                raise ValueError("Invalid category vocabulary")
        self.numeric_names = [n for n in names if n in numeric]
        self.categorical_names = [n for n in names if n in categorical]
        self.output_feature_names = self.numeric_names + [
            f"{name}={category}" for name in self.categorical_names for category in categorical[name]
        ]
        if (
            len(set(self.output_feature_names)) != len(self.output_feature_names)
            or len(self.output_feature_names) != state["output_width"]
        ):
            raise ValueError("Feature output width/name mismatch")
        labels = state.get("class_names", {})
        if labels and set(labels) != {str(i) for i in range(len(labels))}:
            raise ValueError("Class IDs must be contiguous from zero")
        self.class_names = [str(labels[str(i)]) for i in range(len(labels))]
        transform = state.get("target_transform", {"name": "identity"})
        if transform["name"] not in {"identity", "standardize"}:
            raise ValueError("Unsupported frozen target transform")
        if transform["name"] == "standardize" and (
            not np.isfinite([transform["mean"], transform["std"]]).all() or transform["std"] <= 0
        ):
            raise ValueError("Invalid target statistics")

    def to_dict(self):
        return {
            "adapter_version": "phase25.1",
            "state": copy.deepcopy(self.state),
            "output_feature_names": list(self.output_feature_names),
            "class_names": list(self.class_names),
        }

    @classmethod
    def from_dict(cls, document):
        adapter = cls(document["state"])
        if (
            document["adapter_version"] != "phase25.1"
            or document["output_feature_names"] != adapter.output_feature_names
            or document["class_names"] != adapter.class_names
        ):
            raise ValueError("Serialized adapter order/mapping mismatch")
        return adapter

    def transform(self, frame):
        if len(frame.columns) != len(set(frame.columns)) or set(frame.columns) != set(
            self.state["input_columns"]
        ):
            raise ValueError("Feature schema mismatch; IDs/targets must be excluded")
        columns = []
        for name in self.numeric_names:
            rule = self.state["numeric"][name]
            value = (
                pd.to_numeric(frame[name], errors="raise")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(rule["median"])
            )
            columns.append(((value.to_numpy(dtype=float) - rule["mean"]) / rule["std"])[:, None])
        for name in self.categorical_names:
            value = frame[name].fillna("__missing__").astype(str).to_numpy()
            columns.append(value[:, None] == np.asarray(self.state["categorical"][name])[None, :])
        result = np.ascontiguousarray(np.concatenate(columns, axis=1), dtype=np.float32)
        if not np.isfinite(result).all():
            raise ValueError("Nonfinite input tensor")
        return result

    def target(self, values, *, inverse=False):
        result = np.asarray(values, dtype=np.float64)
        if not np.isfinite(result).all():
            raise ValueError("Missing/nonfinite labels must not be imputed")
        rule = self.state.get("target_transform", {"name": "identity"})
        if rule["name"] == "standardize":
            result = result * rule["std"] + rule["mean"] if inverse else (result - rule["mean"]) / rule["std"]
        return result if inverse else result.astype(np.float32)

    def inverse_target(self, values):
        return self.target(values, inverse=True)

    def encode_labels(self, values):
        mapping = {name: i for i, name in enumerate(self.class_names)}
        if not mapping or any(value not in mapping for value in values):
            raise ValueError("Unknown class label")
        return np.asarray([mapping[value] for value in values], dtype=np.int64)



#### image_batch

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def image_batch(images, implementation):
    images = np.asarray(images)
    if images.ndim != 4 or images.shape[-1] != 3 or images.dtype != np.uint8:
        raise ValueError("Images must be uint8 NHWC RGB; resize/decode through frozen source loader")
    if implementation not in {"scratch", "keras", "pytorch"}:
        raise ValueError("Unknown implementation")
    result = images.astype(np.float32) / 255
    if implementation == "pytorch":
        result = result.transpose(0, 3, 1, 2)
    return np.ascontiguousarray(result)



#### select_rows

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def select_rows(source_ids, selected_ids):
    if len(set(source_ids)) != len(source_ids) or len(set(selected_ids)) != len(selected_ids):
        raise ValueError("Duplicate IDs cannot be aligned")
    index = {value: i for i, value in enumerate(source_ids)}
    if any(value not in index for value in selected_ids):
        raise ValueError("Unknown selected ID")
    return [index[value] for value in selected_ids]



#### window_batch

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def window_batch(frame, windows, adapter):
    """Only materialize requested windows; source indices must match Phase24 cache."""
    features = adapter.state["input_columns"]
    x, y = [], []
    for row in windows:
        start, end = row["start_index"], row["origin_index"]
        targets = row["target_indices"]
        if start < 0 or end < start or not targets or min(targets) <= end or max(targets) >= len(frame):
            raise ValueError("Noncausal/out-of-range window")
        context = frame.iloc[start : end + 1]
        future = frame.iloc[targets]
        if not (context.series.eq(row["series"]).all() and future.series.eq(row["series"]).all()):
            raise ValueError("Window crosses series")
        x.append(adapter.transform(context[features]))
        y.append(adapter.target(future.target.to_numpy()))
    if not x:
        raise ValueError("Empty window batch")
    return np.stack(x), np.stack(y)



### `src.ch06_multi_agent.tools`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch06_multi_agent.tools src/ch06_multi_agent/tools.py both
"""Allowlisted frozen NumPy predictors and exact evidence/series lookup tools."""

import copy

import numpy as np

from src.ch05_knowledge_rag.preservation import checked, ref

from src.common.artifact_core import ArtifactStore

from src.common.contracts import ROOT, read_json

from src.common.numpy_engine import NumpyNetwork

from src.common.tensor_adapters import TensorAdapter



#### ToolCatalog

In [ ]:
%%inline_module src.ch06_multi_agent.tools src/ch06_multi_agent/tools.py both
class ToolCatalog:
    def __init__(self, document):
        self.records = document["records"]
        self.models = {}
        if len(self.records) > 200 or len(document["models"]) != 3:
            raise ValueError("Catalog exceeds fixed local budget")
        for domain, spec in document["models"].items():
            config = read_json(checked(spec["config"]))
            metadata_path = checked(spec["checkpoint_metadata"])
            checkpoint_path = checked(spec["checkpoint"])
            metadata = read_json(metadata_path)
            adapter_contract = read_json(checked(config["adapter"]))
            adapter = TensorAdapter(read_json(checked(adapter_contract["preprocessing"])))
            if adapter.output_feature_names != adapter_contract["output_feature_names"]:
                raise ValueError("Canonical feature order changed")
            weights = ArtifactStore(checkpoint_path.parent).load_checkpoint(
                "best", metadata["provenance"], expected_manifest_sha256=spec["checkpoint_metadata"]["sha256"]
            )
            self.models[domain] = (NumpyNetwork(config["plan"], weights), adapter)
        references = {
            item["path"]: item for record in self.records.values() for item in record["artifact_refs"]
        }
        for item in references.values():
            checked(item)

    def call(self, tool, record_id, attempt=0, fault=None):
        if fault == "transient" and attempt == 0:
            return {"status": "error", "code": "transient"}
        if fault == "permanent":
            return {"status": "error", "code": "unavailable"}
        record = self.records.get(record_id)
        if record is None or record["tool"] != tool:
            raise ValueError("Record/tool not authorized")
        result = {
            k: copy.deepcopy(record[k]) for k in ("values", "unit", "text", "citations", "artifact_refs")
        }
        # Common V3 trace paths intentionally exclude apps/. Keep the original citation
        # container hash and point trace refs through its already locked preservation index.
        result["artifact_refs"] = [
            ref(ROOT / "artifacts/manifests/phase30/preserved_snapshot.json")
            if item["path"].startswith("apps/")
            else item
            for item in result["artifact_refs"]
        ]
        if record["operation"] == "numeric":
            model, adapter = self.models[record["domain"]]
            x = np.asarray(record["input"], dtype="float32")[None]
            if not np.isfinite(x).all() or list(x.shape[1:]) != list(model.plan["input_shape"]):
                raise ValueError("Invalid canonical model input")
            raw = model.forward(x).astype("float64")
            if record["domain"] == "diabetes":
                result["values"] = [float(1 / (1 + np.exp(-np.clip(raw[0, 0], -60, 60))))]
            else:
                result["values"] = adapter.inverse_target(raw)[0].reshape(-1).tolist()
            result["text"] = "Frozen NumPy CPU tool output: " + ", ".join(
                format(v, ".8g") for v in result["values"]
            )
        return result



## Thực hành: diabetes

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# Chương6 — Tiểu đường
Chọn kernel WSL ai-v3-torch. TRAIN-only demo, không benchmark/TEST, không lời khuyên chuyên môn.
Code dùng subprocess riêng vì fork executor không chạy trong thread Jupyter. Không gọi API mạng.

In [ ]:
if RUN_EXPERIMENTS:
    import json, subprocess, sys
    from pathlib import Path
    root = Path.cwd().resolve()
    while not (root / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').is_file():
        if root == root.parent: raise RuntimeError('Open from repository tree')
        root = root.parent
    p = subprocess.run([sys.executable, '-m', 'scripts.demo_phase31', '--domain', 'diabetes'], cwd=root, capture_output=True, text=True, check=True)
    demo = json.loads(p.stdout)
    print(json.dumps({v: {'status': r['trace']['status'], 'values': r['values'], 'nodes': len(r['trace']['nodes']), 'citations': r['trace']['citations']} for v,r in demo['outputs'].items()}, ensure_ascii=False, indent=2))


{
  "A0": {
    "status": "succeeded",
    "values": [
      0.25386870685992713
    ],
    "nodes": 1,
    "citations": [
      {
        "evidence_id": "evidence_diabetes_numeric_000",
        "locator": "parent_train/row_0000015",
        "source_sha256": "c53fa5ef792a32926b3b59803bae50159d802c9a67445822aecf0880cb6fbe8e",
        "source_url": "https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators"
      }
    ]
  },
  "A1": {
    "status": "succeeded",
    "values": [
      0.25386870685992713
    ],
    "nodes": 2,
    "citations": [
      {
        "evidence_id": "evidence_diabetes_numeric_000",
        "locator": "parent_train/row_0000015",
        "source_sha256": "c53fa5ef792a32926b3b59803bae50159d802c9a67445822aecf0880cb6fbe8e",
        "source_url": "https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators"
      }
    ]
  },
  "A2": {
    "status": "succeeded",
    "values": [
      0.25386870685992713
    ],
    "nodes": 4,
    "citations": [

## Thực hành: gold

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# Chương6 — Giá vàng
Chọn kernel WSL ai-v3-torch. TRAIN-only demo, không benchmark/TEST, không lời khuyên chuyên môn.
Code dùng subprocess riêng vì fork executor không chạy trong thread Jupyter. Không gọi API mạng.

In [ ]:
if RUN_EXPERIMENTS:
    import json, subprocess, sys
    from pathlib import Path
    root = Path.cwd().resolve()
    while not (root / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').is_file():
        if root == root.parent: raise RuntimeError('Open from repository tree')
        root = root.parent
    p = subprocess.run([sys.executable, '-m', 'scripts.demo_phase31', '--domain', 'gold'], cwd=root, capture_output=True, text=True, check=True)
    demo = json.loads(p.stdout)
    print(json.dumps({v: {'status': r['trace']['status'], 'values': r['values'], 'nodes': len(r['trace']['nodes']), 'citations': r['trace']['citations']} for v,r in demo['outputs'].items()}, ensure_ascii=False, indent=2))


{
  "A0": {
    "status": "succeeded",
    "values": [
      34.041584925525314,
      32.72447998441302,
      35.20187368605272
    ],
    "nodes": 1,
    "citations": [
      {
        "evidence_id": "evidence_gold_numeric_000",
        "locator": "parent_train/GOLD/1961-12-01T00:00:00",
        "source_sha256": "7eca5bf902616687788dbb141c53eb3e82f8e48bd41b956f171d667a444c08ee",
        "source_url": "https://www.worldbank.org/en/research/commodity-markets"
      }
    ]
  },
  "A1": {
    "status": "succeeded",
    "values": [
      34.041584925525314,
      32.72447998441302,
      35.20187368605272
    ],
    "nodes": 2,
    "citations": [
      {
        "evidence_id": "evidence_gold_numeric_000",
        "locator": "parent_train/GOLD/1961-12-01T00:00:00",
        "source_sha256": "7eca5bf902616687788dbb141c53eb3e82f8e48bd41b956f171d667a444c08ee",
        "source_url": "https://www.worldbank.org/en/research/commodity-markets"
      }
    ]
  },
  "A2": {
    "status": "succeeded

## Thực hành: house

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# Chương6 — Giá nhà
Chọn kernel WSL ai-v3-torch. TRAIN-only demo, không benchmark/TEST, không lời khuyên chuyên môn.
Code dùng subprocess riêng vì fork executor không chạy trong thread Jupyter. Không gọi API mạng.

In [ ]:
if RUN_EXPERIMENTS:
    import json, subprocess, sys
    from pathlib import Path
    root = Path.cwd().resolve()
    while not (root / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').is_file():
        if root == root.parent: raise RuntimeError('Open from repository tree')
        root = root.parent
    p = subprocess.run([sys.executable, '-m', 'scripts.demo_phase31', '--domain', 'house'], cwd=root, capture_output=True, text=True, check=True)
    demo = json.loads(p.stdout)
    print(json.dumps({v: {'status': r['trace']['status'], 'values': r['values'], 'nodes': len(r['trace']['nodes']), 'citations': r['trace']['citations']} for v,r in demo['outputs'].items()}, ensure_ascii=False, indent=2))


{
  "A0": {
    "status": "succeeded",
    "values": [
      118115.72406888881
    ],
    "nodes": 1,
    "citations": [
      {
        "evidence_id": "evidence_house_numeric_000",
        "locator": "parent_train/row_0000001",
        "source_sha256": "c6a9010960cc01cff51b663af8f744144c07f427b6c9a3b26a97eaaa60da687f",
        "source_url": "https://jse.amstat.org/v19n3/decock.pdf"
      }
    ]
  },
  "A1": {
    "status": "succeeded",
    "values": [
      118115.72406888881
    ],
    "nodes": 2,
    "citations": [
      {
        "evidence_id": "evidence_house_numeric_000",
        "locator": "parent_train/row_0000001",
        "source_sha256": "c6a9010960cc01cff51b663af8f744144c07f427b6c9a3b26a97eaaa60da687f",
        "source_url": "https://jse.amstat.org/v19n3/decock.pdf"
      }
    ]
  },
  "A2": {
    "status": "succeeded",
    "values": [
      118115.72406888881
    ],
    "nodes": 4,
    "citations": [
      {
        "evidence_id": "evidence_house_numeric_000",
        "

In [ ]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})
